# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"

df = pd.read_csv(url)

# Convert column names to lowercase snake_case
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

# Rename the monthly premium column to match the exercise
df = df.rename(columns={
    "monthly_premium_auto": "monthly_premium"
})

# Remove the unnecessary index column
df = df.drop(columns=["unnamed:_0"], errors="ignore")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (10910, 25)


,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,income,...,number_of_open_complaints,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type
0,DK49336,Arizona,4809.216960,No,Basic,College,2/18/11,Employed,M,48029,...,0.0,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,NaN
1,KX64629,California,2228.525238,No,Basic,College,1/18/11,Unemployed,F,0,...,0.0,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,NaN
2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2/10/11,Employed,M,22139,...,0.0,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A
3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,49078,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,1/17/11,Medical Leave,F,23675,...,NaN,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,NaN


### Conclusion

A total of **1,399 customers** responded "Yes" to the last marketing
campaign and had a total claim amount below $1,000. These customers may
be valuable targets for future marketing because they showed interest
in the campaign while also having relatively low claim amounts.

In [2]:
low_claim_responders = df[
    (df["total_claim_amount"] < 1000) &
    (df["response"] == "Yes")
].copy()

print("Number of selected customers:", len(low_claim_responders))

display(
    low_claim_responders[
        [
            "customer",
            "state",
            "response",
            "total_claim_amount",
            "monthly_premium",
            "customer_lifetime_value"
        ]
    ].head()
)

Number of selected customers: 1399


,customer,state,response,total_claim_amount,monthly_premium,customer_lifetime_value
3,XL78013,Oregon,Yes,484.013411,97,22332.439460
8,FM55990,California,Yes,739.200000,154,5989.773931
15,CW49887,California,Yes,547.200000,114,4626.801093
19,NJ54277,California,Yes,19.575683,94,3746.751625
27,MQ68407,Oregon,Yes,60.036683,111,4376.363592


In [3]:
yes_customers = df[df["response"] == "Yes"].copy()

print("Customers who responded Yes:", len(yes_customers))

Customers who responded Yes: 1466


In [4]:
segment_analysis = (
    yes_customers
    .groupby(["policy_type", "gender"])
    .agg(
        customers=("customer", "count"),
        average_monthly_premium=("monthly_premium", "mean"),
        average_customer_lifetime_value=(
            "customer_lifetime_value", "mean"
        ),
        average_total_claim_amount=("total_claim_amount", "mean"),
        median_total_claim_amount=("total_claim_amount", "median")
    )
    .round(2)
    .reset_index()
)

display(segment_analysis)

,policy_type,gender,customers,average_monthly_premium,average_customer_lifetime_value,average_total_claim_amount,median_total_claim_amount
0,Corporate Auto,F,169,94.30,7712.63,433.74,420.04
1,Corporate Auto,M,154,92.19,7944.47,408.58,364.80
2,Personal Auto,F,540,99.00,8339.79,452.97,424.33
3,Personal Auto,M,536,91.09,7448.38,457.01,412.80
4,Special Auto,F,35,92.31,7691.58,453.28,420.04
5,Special Auto,M,32,86.34,8247.09,429.53,345.60


### Profitability and risk conclusions

Among customers who responded "Yes", female Personal Auto customers
have the highest average monthly premium ($99.00) and the highest
average customer lifetime value among the large customer segments
($8,339.79). However, their average claim amount is also relatively
high at $452.97.

Male Corporate Auto customers appear to provide a favourable balance.
Their average customer lifetime value is $7,944.47, while their average
claim amount of $408.58 is the lowest of all the segments. This may
make them one of the more attractive low-risk segments.

Male Special Auto customers have a high average customer lifetime value
of $8,247.09 and a relatively low average claim amount of $429.53.
However, this segment contains only 32 customers, so the result should
be interpreted carefully.

Male Personal Auto customers have the highest average claim amount
($457.01) and a lower average customer lifetime value ($7,448.38).
Therefore, this segment may appear less attractive than the other large
segments.

These results only provide indicators of profitability. Actual profit
cannot be calculated without additional information such as policy
duration, operating expenses and the time period covered by the claims.

In [5]:
customers_by_state = (
    df.groupby("state")
      .size()
      .reset_index(name="number_of_customers")
      .sort_values("number_of_customers", ascending=False)
)

display(customers_by_state)

,state,number_of_customers
1,California,3552
3,Oregon,2909
0,Arizona,1937
2,Nevada,993
4,Washington,888


In [6]:
states_over_500 = customers_by_state[
    customers_by_state["number_of_customers"] > 500
]

display(states_over_500)

,state,number_of_customers
1,California,3552
3,Oregon,2909
0,Arizona,1937
2,Nevada,993
4,Washington,888


### Conclusion

California has the largest number of customers with 3,552, followed by
Oregon with 2,909 and Arizona with 1,937. All five states represented
in the dataset have more than 500 customers, so all of them remain
after applying the filter.

In [7]:
clv_by_education_gender = (
    df.groupby(["education", "gender"])
      .agg(
          maximum_clv=("customer_lifetime_value", "max"),
          minimum_clv=("customer_lifetime_value", "min"),
          median_clv=("customer_lifetime_value", "median")
      )
      .round(2)
      .reset_index()
)

display(clv_by_education_gender)

,education,gender,maximum_clv,minimum_clv,median_clv
0,Bachelor,F,73225.96,1904.00,5640.51
1,Bachelor,M,67907.27,1898.01,5548.03
2,College,F,61850.19,1898.68,5623.61
3,College,M,61134.68,1918.12,6005.85
4,Doctor,F,44856.11,2395.57,5332.46
5,Doctor,M,32677.34,2267.60,5577.67
6,High School or Below,F,55277.45,2144.92,6039.55
7,High School or Below,M,83325.38,1940.98,6286.73
8,Master,F,51016.07,2417.78,5729.86
9,Master,M,50568.26,2272.31,5579.10


### Conclusion

Male customers with a High School education or below have the highest
maximum customer lifetime value at $83,325.38. This group also has the
highest median customer lifetime value at $6,286.73.

Female customers with a Doctor-level education have the lowest median
customer lifetime value at $5,332.46. However, the differences between
most median values are relatively small.

The maximum values are much higher than the corresponding medians,
which suggests that customer lifetime value contains extreme high-value
customers or outliers. Therefore, the median provides a more
representative measure of the typical customer than the maximum.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [8]:
# Convert the column to datetime
df["effective_to_date"] = pd.to_datetime(
    df["effective_to_date"],
    format="%m/%d/%y"
)

# Extract the month name
df["month"] = df["effective_to_date"].dt.month_name()

df[["effective_to_date", "month"]].head()

,effective_to_date,month
0,2011-02-18,February
1,2011-01-18,January
2,2011-02-10,February
3,2011-01-11,January
4,2011-01-17,January


In [9]:
policies_by_state_month = (
    df.groupby(["state", "month"])
      .size()
      .reset_index(name="policies_sold")
)

policies_table = policies_by_state_month.pivot(
    index="state",
    columns="month",
    values="policies_sold"
).fillna(0).astype(int)

# Arrange the available months chronologically
month_order = ["January", "February"]

policies_table = policies_table.reindex(
    columns=month_order,
    fill_value=0
)

display(policies_table)

month,January,February
state,,
Arizona,1008,929
California,1918,1634
Nevada,551,442
Oregon,1565,1344
Washington,463,425


### Conclusion

California recorded the highest number of policies sold in both
January and February, followed by Oregon and Arizona. Every state sold
more policies in January than in February, indicating an overall
decrease in policy sales during February.